# Biomedical LLM Adaptation — Colab T4 Runner

Trains a 4-bit **QLoRA** adapter for `Qwen2.5-7B-Instruct` on **MedMCQA**, then scores base vs fine-tuned on MedMCQA (in-domain) and PubMedQA (out-of-domain) with the EleutherAI lm-eval-harness.

**Storage:** everything stays on Colab's local disk — **nothing is written to Google Drive.** Local disk is wiped when the runtime recycles, so *Step 4* persists results off-Colab: the adapter goes to the **HF Hub** and the result tables download to **your laptop**.

**T4 notes:** Turing (compute 7.5) → fp16, no bf16. The 5K slice fits one free session; for a long run (20K+) finish in one sitting, since local-only storage has no checkpoint safety net.

> Set **Runtime → Change runtime type → T4 GPU** before running.

## Setup

In [ ]:
# GPU check
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# Clone the project (or fast-forward it if already cloned), then enter it.
REPO_URL = 'https://github.com/Udit013/biomed-llm-peft.git'
import os
if os.path.isdir('biomed-llm-peft/.git'):
    !git -C biomed-llm-peft pull --ff-only
else:
    !git clone $REPO_URL biomed-llm-peft
%cd biomed-llm-peft

In [ ]:
# Install pinned deps. Colab ships a CUDA-matched torch -> do NOT reinstall it.
# bitsandbytes must match Colab's torch/CUDA, so install the latest (an old pin
# breaks on Colab's modern Triton: "No module named 'triton.ops'").
!grep -vE '^(torch==|bitsandbytes)' requirements.txt > /tmp/reqs.txt
!pip install -q -r /tmp/reqs.txt
!pip install -q -U bitsandbytes
!python -c "import bitsandbytes as bnb; print('bitsandbytes', bnb.__version__)"
print('deps installed')

In [ ]:
# Local-only config. outputs/ and results/ live on Colab's local disk; the HF
# model cache stays local too (never Drive). Optional tokens below are blank by
# default — training/eval run fine without them.
import os
os.environ.pop('HF_HOME', None)                 # model cache -> local disk, not Drive
for d in ['outputs', 'results']:
    os.makedirs(d, exist_ok=True)

os.environ['WANDB_API_KEY'] = ''                # blank => local JSONL logging only
os.environ['WANDB_PROJECT'] = 'biomed-llm-peft'
os.environ['HF_TOKEN']      = ''                # needed only to push the adapter (Step 4)

!python -m src.utils.env

## Step 1 — Train the 5K QLoRA adapter

~70 min on a T4. Resumes from a local checkpoint if one exists in this session.

In [ ]:
!python scripts/train.py --config configs/qlora_5k.yaml

## Step 2 — Evaluate (base vs fine-tuned)

Base 0-shot and the adapter are scored at the **same** `EVAL_LIMIT` so the comparison is apples-to-apples. `EVAL_LIMIT = None` runs the full official split (needs Colab Pro / a longer GPU).

In [ ]:
EVAL_LIMIT = 200                                # items per task; None = full split
lim = f'--limit {EVAL_LIMIT}' if EVAL_LIMIT else ''
!python scripts/run_eval.py --mode base0 --output results/base_0shot $lim
!python scripts/run_eval.py --adapter outputs/qlora_5k --output results/qlora_5k $lim
# (optional, slow ~3h) base 5-shot:
# !python scripts/run_eval.py --mode base5 --output results/base_5shot $lim

## Step 3 — Results

Per-subject error analysis + the headline data-scaling table, filled with the real numbers from Step 2.

In [ ]:
!python scripts/error_analysis.py --base results/base_0shot --finetuned results/qlora_5k --out results/error_analysis_qlora_5k
!python scripts/results_table.py --out results/headline_table.md
print(open('results/headline_table.md').read())

## Step 4 — Persist off-Colab (no Drive)

Local disk is ephemeral, so save what matters elsewhere: the adapter to the HF Hub, the result tables to your laptop.

In [ ]:
# Push the LoRA adapter + model card to the HF Hub (set HF_TOKEN in Setup first).
# Skips gracefully with no token.
!python scripts/push_to_hub.py --adapter outputs/qlora_5k \
    --repo-id Udit013/qwen2.5-7b-medmcqa-qlora-5k \
    --base-eval results/base_0shot --ft-eval results/qlora_5k

In [ ]:
# Download the result tables to your laptop (browser Downloads). To also grab the
# 154 MB adapter, change `results` to `results outputs/qlora_5k`.
from google.colab import files
!zip -qr /content/biomed_results.zip results
files.download('/content/biomed_results.zip')

## Optional — Inference cost

Quantized (4-bit) throughput/latency/VRAM on the T4. vLLM auto-skips on T4. Adding `base lora_fp16` measures fp16 too, but a 7B in fp16 doesn't fit 16 GB and will CPU-offload (slow).

In [ ]:
!python scripts/inference_cost.py --adapter outputs/qlora_5k --configs quantized vllm

## Scaling up

To get the data-scaling curve, re-run **Steps 1–3** with `configs/qlora_20k.yaml` (fits one free session). 50K/full need Colab Pro or a longer/cloud GPU.